<a href="https://colab.research.google.com/github/polis-community/red-dwarf/blob/main/docs/notebooks/agora-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet red-dwarf[all]@git+https://github.com/polis-community/red-dwarf.git@main

# Agora Pipeline Demo

This notebook demonstrates the **Agora pipeline**, the recommended default for Red Dwarf.

Agora builds on the same core as Polis (PCA + KMeans) but improves statement selection and consensus scoring:

- **Benjamini-Hochberg FDR control** for adaptive, data-driven statement selection (replaces Polis's heuristic `pick_max=5`)
- **Simes' p-value combination** for repness (valid under positive dependence)
- **Effective agreement GAC**: `prod(pa*(1-pd))^(1/n)` penalizes divided groups
- **Zero-vote filtering** to avoid inflating hypothesis counts

## Load Data

We use a local fixture with <100 participants from a real Polis conversation about tech and politics in 2018.

In [ ]:
from reddwarf.data_loader import Loader
from reddwarf.utils.statements import process_statements

# You can also load from a Polis URL: Loader(polis_id="r2dfw8eambusb8buvecjt")
loader = Loader(filepaths=[
    "../../tests/fixtures/below-100-ptpts/votes.json",
    "../../tests/fixtures/below-100-ptpts/comments.json",
    "../../tests/fixtures/below-100-ptpts/conversation.json",
])

_, _, mod_out_statement_ids, meta_statement_ids = process_statements(
    statement_data=loader.comments_data
)

# Build a lookup for statement text
stmt_text = {}
for c in loader.comments_data:
    sid = c.get("tid", c.get("statement_id"))
    stmt_text[sid] = c.get("txt", "")

print(f"Loaded {len(loader.votes_data)} votes")
print(f"Statements: {len(loader.comments_data)} total, {len(mod_out_statement_ids)} moderated out")

## Run the Agora Pipeline

In [ ]:
from reddwarf.implementations.agora import run_pipeline

result = run_pipeline(
    votes=loader.votes_data,
    mod_out_statement_ids=mod_out_statement_ids,
    meta_statement_ids=meta_statement_ids,
    random_state=42,
)

n_groups = len(result.ranked_repness)
n_clustered = len(result.participants_df[result.participants_df["to_cluster"]])
n_statements = len(result.statements_df)

print(f"Participants: {n_clustered} clustered into {n_groups} groups")
print(f"Statements: {n_statements - len(mod_out_statement_ids)} active ({len(mod_out_statement_ids)} moderated out)")

## Representative Statements (Repness)

For each group, statements are ranked by **effect size** (`ra * pa` for agree, `rd * pd` for disagree) and selected using Benjamini-Hochberg FDR control.

The `[*]` marker indicates BH-selected statements.

In [ ]:
def truncate(text, max_len=60):
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."

for gid in sorted(result.ranked_repness.keys()):
    print(f"\n--- Group {gid} ---")
    print(f"{'#':>2}  Sel  Dir       Effect  Statement")
    for s in result.ranked_repness[gid]:
        sel = "[*]" if s.selected else "[ ]"
        direction = "agree" if s.repful_for == "agree" else "disagree"
        text = truncate(stmt_text.get(s.statement_id, "?"))
        print(f"{s.rank:>2}  {sel}  {direction:<8}  {s.effect_size:.4f}  s{s.statement_id}: {text}")

## Cross-Group Comparison

The `group_comment_stats` DataFrame lets you compare how different groups responded to each statement.

In [ ]:
group_ids = sorted(result.ranked_repness.keys())
stats = result.group_comment_stats

# Show top repness statement for group 0 across all groups
top_stmt = result.ranked_repness[0][0]
sid = top_stmt.statement_id
print(f"Statement s{sid}: {truncate(stmt_text.get(sid, '?'))}")
print(f"Direction: {top_stmt.repful_for}, Effect size: {top_stmt.effect_size:.4f}")
print()
for g in group_ids:
    row = stats.loc[(g, sid)]
    print(f"  Group {g}: pa={row['pa']:.2f}, pd={row['pd']:.2f} (agree={int(row['na'])}, disagree={int(row['nd'])}, seen={int(row['ns'])})")

## Group-Aware Consensus (GAC)

Agora uses **effective agreement**: `prod(pa * (1-pd))^(1/n_groups)`.

Unlike raw Polis GAC (`prod(pa)^(1/n)`), this penalizes groups that are genuinely divided (high agree AND high disagree).

In [ ]:
gac = result.group_aware_consensus

# Top statements by effective agreement (agree direction)
sorted_agree = sorted(
    [(sid, score) for sid, score in gac["agree"].items() if sid not in mod_out_statement_ids],
    key=lambda x: x[1], reverse=True,
)

print("Top 10 statements by group-aware consensus (agree):")
print(f"{'#':>2}  Stmt  GAC     Text")
for rank, (sid, score) in enumerate(sorted_agree[:10], 1):
    text = truncate(stmt_text.get(sid, "?"))
    print(f"{rank:>2}  s{sid:<3}  {score:.4f}  {text}")

## Consensus Statements

Consensus statements use BH FDR control on the one-prop test z-scores, ranked by pa (agree) or pd (disagree).

In [ ]:
print("--- Consensus (agree) ---")
print(f"{'#':>2}  Sel  Stmt  Effect  p_adj    Text")
for s in result.ranked_consensus.agree:
    sel = "[*]" if s.selected else "[ ]"
    text = truncate(stmt_text.get(s.statement_id, "?"))
    print(f"{s.rank:>2}  {sel}  s{s.statement_id:<3}  {s.effect_size:.4f}  {s.adjusted_p_value:.4f}  {text}")

print()
print("--- Consensus (disagree) ---")
print(f"{'#':>2}  Sel  Stmt  Effect  p_adj    Text")
for s in result.ranked_consensus.disagree:
    sel = "[*]" if s.selected else "[ ]"
    text = truncate(stmt_text.get(s.statement_id, "?"))
    print(f"{s.rank:>2}  {sel}  s{s.statement_id:<3}  {s.effect_size:.4f}  {s.adjusted_p_value:.4f}  {text}")

## Comparison with Polis Selection

For reference, here's what stock Polis selection (`pick_max=5`) produces on the same data.

In [ ]:
from reddwarf.utils.stats import select_representative_statements
from reddwarf.utils.consensus import select_consensus_statements

polis_repness = select_representative_statements(
    grouped_stats_df=result.group_comment_stats,
    mod_out_statement_ids=mod_out_statement_ids,
)
polis_consensus = select_consensus_statements(
    vote_matrix=result.raw_vote_matrix,
    mod_out_statement_ids=mod_out_statement_ids,
)

def fmt_stmt(sid, direction):
    d = "a" if direction == "agree" else "d"
    return f"s{sid}({d})"

print("--- Repness comparison ---")
for gid in sorted(result.ranked_repness.keys()):
    polis_stmts = [fmt_stmt(s["tid"], s["repful-for"]) for s in polis_repness.get(gid, [])]
    agora_stmts = [fmt_stmt(s.statement_id, s.repful_for) for s in result.ranked_repness[gid] if s.selected]
    print(f"Group {gid}:")
    print(f"  Polis (pick_max=5): {' '.join(polis_stmts) if polis_stmts else '(none)'}")
    print(f"  Agora (BH fdr=0.10): {' '.join(agora_stmts) if agora_stmts else '(none)'}")

print()
print("--- Consensus comparison ---")
polis_agree = [f"s{s['tid']}" for s in polis_consensus["agree"]]
agora_agree = [f"s{s.statement_id}" for s in result.ranked_consensus.agree if s.selected]
print(f"Agree:")
print(f"  Polis: {' '.join(polis_agree) if polis_agree else '(none)'}")
print(f"  Agora: {' '.join(agora_agree) if agora_agree else '(none)'}")

polis_disagree = [f"s{s['tid']}" for s in polis_consensus["disagree"]]
agora_disagree = [f"s{s.statement_id}" for s in result.ranked_consensus.disagree if s.selected]
print(f"Disagree:")
print(f"  Polis: {' '.join(polis_disagree) if polis_disagree else '(none)'}")
print(f"  Agora: {' '.join(agora_disagree) if agora_disagree else '(none)'}")

## Key Differences Summary

| Feature | Polis | Agora |
|---------|-------|-------|
| **Statement selection** | Heuristic `pick_max=5` per group | BH FDR control (adaptive, data-driven) |
| **P-value combination** | z-score thresholds | Simes' method (valid under positive dependence) |
| **Group-aware consensus** | `prod(pa)^(1/n)` (raw agreement) | `prod(pa*(1-pd))^(1/n)` (penalizes divided groups) |
| **Zero-vote handling** | Included in hypothesis testing | Excluded from BH hypothesis count |
| **Output** | Only top-N selected statements | ALL statements ranked with selection flags |

**Use Agora** for production analysis — it provides statistically rigorous, adaptive selection.

**Use Polis** when you need to exactly reproduce upstream Polis behavior for verification or comparison.